# Sprint 1 ingestion pipeline (extract → clean → chunk)

Reproducible runner for putting a single document through the pipeline.
Point the two paths in the **Config** cell at the document you want, then run the
cells top to bottom:

1. **Extract** — `extract.py` calls the Unstructured.io Jobs/VLM API and saves raw
   JSON into `data/processed/<doc>/`. *(Live, paid API call.)*
2. **Clean** — `clean.py` turns that raw JSON into `<doc>-CLEANED.json` (for
   chunking) + `<doc>-CLEANED-preview.txt` (for reading).
3. **Chunk** — the groupmate's chunking script runs on the cleaned JSON.

The `pcbus_working_together` document is already done — use a different document
here.

## Config — choose the document

In [1]:
import sys
from pathlib import Path

# Resolve the repo root so `src` imports work no matter where Jupyter started.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# --- add chosen filepath here (two explicit paths) ---
# input : the raw PDF folder under data/raw/<category>/<doc>
# output: where the JSON is written under data/processed/<doc>
INPUT_DIR = ROOT / "data/raw/building_and_construction/exposure_monitoring_and_health_monitoring"
OUTPUT_DIR = ROOT / "data/processed/exposure-monitoring-and-health-monitoring"

print("Repo root :", ROOT)
print("Input dir :", INPUT_DIR)
print("Output dir:", OUTPUT_DIR)

Repo root : /Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI
Input dir : /Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI/data/raw/building_and_construction/exposure_monitoring_and_health_monitoring
Output dir: /Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI/data/processed/exposure-monitoring-and-health-monitoring


## Step 1 — Extract (Unstructured.io Jobs / VLM API)

Live, paid API call. Submits a job, polls until it finishes, and saves the raw
`*.pdf.json` into `OUTPUT_DIR`. Needs `UNSTRUCTURED_API_KEY` in `.env`.

In [2]:
from src.ingestion import extract
# this cell takes a while to run, as it calls the Unstructured API to extract the document. it can take up to 10 minutes for large documents. 
# the output is saved to OUTPUT_DIR as a JSON file, which can be used in subsequent steps of the pipeline.
saved = extract.extract_document(INPUT_DIR, OUTPUT_DIR)
print("\nRaw JSON saved:")
for p in saved:
    print(" ", p)

Submitting 1 file(s) from /Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI/data/raw/building_and_construction/exposure_monitoring_and_health_monitoring...


INFO: HTTP Request: POST https://platform-api.transform.unstructured.io/api/v1/jobs/ "HTTP/1.1 200 OK"


Job ID: ede56178-121d-4e04-b60d-21eed251a19c


INFO: HTTP Request: GET https://platform-api.transform.unstructured.io/api/v1/jobs/ede56178-121d-4e04-b60d-21eed251a19c "HTTP/1.1 200 OK"


Job status: SCHEDULED


KeyboardInterrupt: 

## Step 2 — Clean (deterministic, no API call)

Reads the raw JSON from `OUTPUT_DIR` and writes `<doc>-CLEANED.json` +
`<doc>-CLEANED-preview.txt` into the same folder. Re-runnable any time.

In [ ]:
from src.ingestion import clean

report = clean.clean_document(OUTPUT_DIR)
report

In [ ]:
from pathlib import Path

p = Path(INPUT_DIR)
print("exists:", p.exists())
print("is_dir:", p.is_dir())

pdfs = list(p.rglob("*.pdf")) + list(p.rglob("*.PDF"))
print("PDFs found:", len(pdfs))
for f in pdfs:
    print(" ", f)

exists: True
is_dir: True
PDFs found: 1
  /Users/rupertguppy/Desktop/R&D-Project/Health-Safety-AI/data/raw/building_and_construction/exposure_monitoring_and_health_monitoring/exposure-monitoring-and-health-monitoring-gpg.pdf


## Step 3 — Chunk (groupmate's script)

Runs on the cleaned **JSON** from Step 2. The chunking module isn't on this branch
yet (it lives on `Luca-Branch`), so this cell is a placeholder — wire it up once the
chunker is merged.

In [ ]:
# TODO: enable once the chunking module is merged from Luca-Branch.
#
#     from src.ingestion import chunk
#     cleaned_json = next(OUTPUT_DIR.glob("*-CLEANED.json"))
#     chunks = chunk.chunk_document(cleaned_json)
#     print(f"{len(chunks)} chunks written")